# Chiffrement hybride RSA-KEM multi-utilisateurs

**A compléter:** DA-COSTA Tom, VIALE Jean-Jacques

Le TP noté comporte des fonctions à programmer et à vérifier. Elles ont quasiment toutes été vues dans les TP précédents. Merci de répondre directement sur la feuille `jupyter` pour les fonctions et d'ajouter des cellules pour valider le fonctionnement de vos fonctions. En l'absence de cellules de validation, la note de l'exercice sera divisée par deux.

On vous fournit le fichier `root_CA.crt` de l'autorité de certification racine, le fichier `CA.pem` de sa biclé et le certificat `Xavier.crt`.

Vous rendrez une archive au format `zip` sur la boîte de dépôt `Moodle` avant le 30 avril. L'archive comprendra:

- la feuille `jupyter` complétée avec les fonctions et les tests de bon fonctionnement;
- les certificats et les biclés correspondantes d'au moins deux utilisateurs (trois serait mieux);
- la biclé correspondant au certificat de Xavier avec un exemple documenté de son usurpation d'identité

## Importation des librairies

In [5]:
import zlib, binascii, secrets, os, base64, pickle, datetime
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.asymmetric import padding, rsa, utils
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.x509 import Certificate, DNSName, load_pem_x509_certificate
from datetime import datetime, timezone, timedelta
from cryptography.exceptions import InvalidTag, InvalidSignature
from cryptography.hazmat.primitives import hashes
from datetime import datetime, timedelta, timezone

## 1. Les clés publiques et gestion de la PKI

Nous allons gérer plusieurs utilisateurs qui devront garantir la relation entre leur identité et leur clé publique RSA assurée par une chaîne de certification qui utilise un certificat racine (fourni) et une autorité de certification intermédiaire (à créer). On s'inspire pour cette partie du **TD3** et du **TD6**.

### 1.1 Construction de la chaîne de certification

**Exercice 1.** Ecrivez la fonction `genRSA` qui prend en entrée la taille en bits de la clé RSA et qui retourne l'objet de clé privée correspondant. Utilisez-la pour engendrer une clé RSA de 2048 bits pour l'autorité de certification intermédiaire de nom `intermed`.

In [ ]:
def genRSA(taille:int)->rsa.RSAPrivateKey:
    if taille < 1024 or taille % 256 != 0:
        raise ValueError("La taille RSA doit etre >= 1024 et multiple de 256")
    return rsa.generate_private_key(public_exponent=65537, key_size=taille)

# Cle privee RSA 2048 bits de l'autorite intermediaire "intermed"
intermed_sk = genRSA(2048)
intermed_pk = intermed_sk.public_key()

In [7]:
# Validation Exercice 1
print(type(intermed_sk).__name__)
print("Taille cle intermediaire:", intermed_sk.key_size, "bits")

RSAPrivateKey
Taille cle intermediaire: 2048 bits


On utilise une chaîne de certification dont la racine vous est fournie `root_CA.crt` avec sa biclé `CA.pem`.
Il faut tout d'abord lire le fichier du certificat racine pour récupérer l'objet certificat correspondant ainsi que la biclé associée.

**Exercice 2.** Ecrivez la fonction `readRSA` qui prend comme entrée un nom de fichier (d'extension `.pem` ou `.crt`). Elle va lire le fichier au format `PEM` ou `PEM X509` et reconstruire l'objet `Cryptography` correspondant (clé publique, clé privée ou certificat). 

In [ ]:
def readRSA(fic_cle:str):
    with open(fic_cle, "rb") as f:
        data = f.read()

    # On tente d'abord certificat, puis cle privee, puis cle publique.
    try:
        return x509.load_pem_x509_certificate(data)
    except ValueError:
        pass

    try:
        return serialization.load_pem_private_key(data, password=None)
    except ValueError:
        pass

    try:
        return serialization.load_pem_public_key(data)
    except ValueError as e:
        raise ValueError(f"Format PEM/CRT non reconnu pour: {fic_cle}") from e

In [11]:
# Validation Exercice 2: lecture des objets racine
root_CA = readRSA("root_CA.crt")
root_CA_sk = readRSA("CA.pem")
print(type(root_CA).__name__)
print(type(root_CA_sk).__name__)

Certificate
RSAPrivateKey


**Exercice 3.** En utilisant le certificat racine `root_CA`, construisez l'objet certificat `cert` pour l'autorité intermédiaire correspondant à la clé vous venez d'engendrer.

In [12]:
# Exercice 3: construction du certificat intermediaire signe par la racine
intermed_subject = x509.Name(
    [
        x509.NameAttribute(NameOID.COUNTRY_NAME, "FR"),
        x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, "PACA"),
        x509.NameAttribute(NameOID.LOCALITY_NAME, "Sophia"),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, "intermed"),
    ]
)

cert_intermediaire = (
    x509.CertificateBuilder()
    .subject_name(intermed_subject)
    .issuer_name(root_CA.subject)
    .public_key(intermed_pk)
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.now(timezone.utc) - timedelta(minutes=1))
    .not_valid_after(datetime.now(timezone.utc) + timedelta(days=3650))
    .add_extension(x509.BasicConstraints(ca=True, path_length=0), critical=True)
    .add_extension(
        x509.KeyUsage(
            digital_signature=True,
            key_encipherment=False,
            content_commitment=False,
            data_encipherment=False,
            key_agreement=False,
            key_cert_sign=True,
            crl_sign=True,
            encipher_only=False,
            decipher_only=False,
        ),
        critical=True,
    )
    .sign(private_key=root_CA_sk, algorithm=hashes.SHA256())
)

print("Certificat intermediaire genere")

Certificat intermediaire genere


Il sera plus facile d'enregistrer votre objet certificat pour le réutiliser par la suite.

**Exercice 4.** Ecrivez la fonction `saveRSA` qui prend en entrée un objet (une clé RSA -privée ou publique- ou un certificat) et un nom de fichier (d'extension `.pem` ou `.crt`). La fonction enregistre l'objet au format `PEM` dans le fichier spécifié. La clé privée sera au format `PKCS8` et la publique au format `PKCS1`. Le certificat n'a pas besoin d'une serialisation particulière. 

Utilisez cette fonction pour enregistrer l'objet biclé puis l'objet certificat intermédiaire.

In [ ]:
def saveRSA(obj, fic_cle:str):
    if isinstance(obj, rsa.RSAPrivateKey):
        data = obj.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption(),
        )
    elif isinstance(obj, rsa.RSAPublicKey):
        data = obj.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.PKCS1,
        )
    elif isinstance(obj, x509.Certificate):
        data = obj.public_bytes(serialization.Encoding.PEM)
    else:
        raise TypeError("Type d'objet non supporte par saveRSA")

    with open(fic_cle, "wb") as f:
        f.write(data)

In [14]:
# Exercice 4: sauvegarde de la bicle et du certificat intermediaire
saveRSA(intermed_sk, "intermed.pem")
saveRSA(intermed_pk, "intermed_pub.pem")
saveRSA(cert_intermediaire, "intermed.crt")
print("Fichiers intermediaires enregistres")

Fichiers intermediaires enregistres


Pour les réutiliser par la suite, on lit ces deux fichiers:

In [15]:
# Relecture des fichiers intermediaires
intermed_sk_loaded = readRSA("intermed.pem")
intermed_pk_loaded = readRSA("intermed_pub.pem")
cert_intermediaire_loaded = readRSA("intermed.crt")
print(type(intermed_sk_loaded).__name__, "|", type(cert_intermediaire_loaded).__name__)

RSAPrivateKey | Certificate


Utilisons cette chaîne de certification pour garantir la relation entre un utilisateur et sa clé.

**Exercice 5.** Ecrivez la fonction `certifie` qui prend en entrée un nom d'utilisateur, sa clé privée et le certificat intermédiaire et qui retourne un objet certificat qui garantit la relation entre l'identité de l'utilisateur, sa clé publique délivrée par l'autorité intermédiaire. Pour simplifier, on supposera qu'une partie des champs X509 sont communs à tous les utilisateurs et que seul le champ `ORGANIZATION_NAME` sera distinct et instancié par le nom de l'utilisateur.

`COUNTRY_NAME, "FR"`
`STATE_OR_PROVINCE_NAME, "PACA"`
`LOCALITY_NAME, "Sophia"`


In [19]:
def certifie(nom:str, pk:rsa.RSAPublicKey, cert_intermediaire:Certificate, bicle_intermediaire:rsa.RSAPrivateKey):
    sujet = x509.Name(
        [
            x509.NameAttribute(NameOID.COUNTRY_NAME, "FR"),
            x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, "PACA"),
            x509.NameAttribute(NameOID.LOCALITY_NAME, "Sophia"),
            x509.NameAttribute(NameOID.ORGANIZATION_NAME, nom),
        ]
    )

    cert_user = (
        x509.CertificateBuilder()
        .subject_name(sujet)
        .issuer_name(cert_intermediaire.subject)
        .public_key(pk)
        .serial_number(x509.random_serial_number())
        .not_valid_before(datetime.now(timezone.utc) - timedelta(minutes=1))
        .not_valid_after(datetime.now(timezone.utc) + timedelta(days=365))
        .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
        .add_extension(
            x509.KeyUsage(
                digital_signature=True,
                key_encipherment=True,
                content_commitment=False,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=False,
                crl_sign=False,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .sign(private_key=bicle_intermediaire, algorithm=hashes.SHA256())
    )
    return cert_user

In [17]:
# Validation Exercice 5: certification d'un utilisateur
alice_sk = genRSA(1024)
alice_cert = certifie("Alice", alice_sk.public_key(), cert_intermediaire_loaded, intermed_sk_loaded)

# Verification cryptographique de la signature du certificat utilisateur
cert_intermediaire_loaded.public_key().verify(
    alice_cert.signature,
    alice_cert.tbs_certificate_bytes,
    padding.PKCS1v15(),
    alice_cert.signature_hash_algorithm,
)
print("Certificat utilisateur Alice valide (signe par l'intermediaire)")

Certificat utilisateur Alice valide (signe par l'intermediaire)


Tout est maintenant en place pour certifier la relation entre un utilisateur et sa clé publique !

### 1.2 Création des utilisateurs et de leurs clés certifiées

**Exercice 6.** Créez à présent les clés `RSA1024` de trois utilisateurs (*e.g.* Alice, Bob et Charles), certifiez-les et enregistrez leurs biclés et leurs certificats. Vérifiez également la validité des certificats crées au moyen d'une fonction `teste_chaine` qui rend `vrai` si la chaîne de certification est valide et `faux` sinon.

In [27]:
def teste_chaine(racine_cert:Certificate, cert_intermediaire:Certificate, cert_utilisateur:Certificate):
    try:
        # 1) La racine doit signer le certificat intermediaire.
        racine_cert.public_key().verify(
            cert_intermediaire.signature,
            cert_intermediaire.tbs_certificate_bytes,
            padding.PKCS1v15(),
            cert_intermediaire.signature_hash_algorithm,
        )

        # 2) L'intermediaire doit signer le certificat utilisateur.
        cert_intermediaire.public_key().verify(
            cert_utilisateur.signature,
            cert_utilisateur.tbs_certificate_bytes,
            padding.PKCS1v15(),
            cert_utilisateur.signature_hash_algorithm,
        )

        # 3) Verification de la relation d'emetteur/sujet dans la chaine.
        if cert_intermediaire.issuer != racine_cert.subject:
            return False
        if cert_utilisateur.issuer != cert_intermediaire.subject:
            return False

        return True
    except Exception:
        return False

In [28]:
# Validation Exercice 6: creation de 3 utilisateurs, certification et sauvegarde
utilisateurs = ["Alice", "Bob", "Charles"]
certificats_users = {}

for nom in utilisateurs:
    sk_u = genRSA(1024)
    pk_u = sk_u.public_key()
    cert_u = certifie(nom, pk_u, cert_intermediaire_loaded, intermed_sk_loaded)

    saveRSA(sk_u, f"{nom}.pem")
    saveRSA(pk_u, f"{nom}_pub.pem")
    saveRSA(cert_u, f"{nom}.crt")

    ok = teste_chaine(root_CA, cert_intermediaire_loaded, cert_u)
    certificats_users[nom] = {"sk": sk_u, "pk": pk_u, "cert": cert_u, "chaine_ok": ok}
    print(f"{nom}: chaine valide = {ok}")

Alice: chaine valide = True
Bob: chaine valide = True
Charles: chaine valide = True


Il faut maintenant pouvoir utiliser ces clés pour chiffrer et déchiffrer.

### 1.3 Chiffrer et déchiffrer avec RSA

**Exercice 7.** Ecrivez les fonctions `encRSA` et `decRSA`  analogues à celles écrites dans le **TD3** avec le padding `PKCS1v15`. Vérifiez-en le bon fonctionnement. 

In [ ]:
def encRSA(octets:bytes, clepub:rsa.RSAPublicKey)->bytes:
    return clepub.encrypt(octets, padding.PKCS1v15())

def decRSA(octets:bytes, clepriv:rsa.RSAPrivateKey)->bytes:
    return clepriv.decrypt(octets, padding.PKCS1v15())

In [23]:
# Validation Exercice 7: chiffrement/dechiffrement RSA PKCS1v15
message_test = b"Test RSA Exercice 7"
bob_pk = certificats_users["Bob"]["pk"]
bob_sk = certificats_users["Bob"]["sk"]

c = encRSA(message_test, bob_pk)
m = decRSA(c, bob_sk)

print("Longueur chiffre:", len(c), "octets")
print("Message dechiffre identique:", m == message_test)

Longueur chiffre: 128 octets
Message dechiffre identique: True


## 2. Petites fonctions préliminaires

### 2.1 Génération aléatoire du secret initial

On s'inspire du **TD1** pour engendrer une valeur aléatoire. Vous utiliserez indifféremment l'extraction de bits de `/dev/random` comme dans le **TD1** ou la librairie `secrets` de `Python`.

**Exercice 8.** Ecrivez la fonction `genrand` qui prend en entrée une taille en bits et qui retourne un secret aléatoire de la taille souhaitée. 

In [1]:
def genrand(bits:int)->bytes:
    if bits <= 0:
        raise ValueError("La taille en bits doit etre positive")

    taille_octets = (bits + 7) // 8
    secret = secrets.token_bytes(taille_octets)
    bits_excedentaires = taille_octets * 8 - bits
    if bits_excedentaires:
        secret = bytes([secret[0] & (0xFF >> bits_excedentaires)]) + secret[1:]
    return secret

### 2.2 Compression des clairs

La compression est souvent une étape préalable au chiffrement. On utilise la librairie `zlib`.

**Exercice 9.** Ecrivez une fonction `compresse` qui prend en entrée une chaîne de caractères `utf-8`, la convertit au format `bytestream` et qui retourne la chaîne de caractères comprimée. Ecrivez ensuite la fonction `decompresse` qui prend en entrée le compressé au format `bytestream` et restitue la chaîne originale au format `utf-8`.

In [2]:
def compresse(texte:str)-> bytes:
    return zlib.compress(texte.encode("utf-8"))


def decompresse(comprime:bytes)->str:
    return zlib.decompress(comprime).decode("utf-8")

### 2.3 Dérivation de clé

La fonction qui permet de dériver une clé a été vue dans le **TD5**. Elle s'inspire directement de la [documentation](https://cryptography.io/en/latest/hazmat/primitives/key-derivation-functions/).

**Exercice 10.** Ecrivez la fonction `derive` qui prend en entrée un secret initial (*e.g.* de 96 bits) et la taille de la clé dérivée (en bits). Elle retourne la clé dérivée du secret initial de la taille spécifiée. Les paramètres de la fonction de dérivation sont les premiers 128 bits du secret initial et les 64 bits suivants constituent le sel.

In [7]:
def derive(secret:bytes, bits:int)->bytes:
    if bits <= 0 or bits % 8 != 0:
        raise ValueError("La taille derivee doit etre positive et multiple de 8")
    if len(secret) == 0:
        raise ValueError("Le secret ne peut pas etre vide")

    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=bits // 8,
        salt=secret[16:24],
        iterations=100000,
    )
    return kdf.derive(secret[:16])

In [8]:
# Validation exercices 8 a 10
secret_test = genrand(192)
texte_test = "Bonjour, securite !"
compresse_test = compresse(texte_test)
cle_test = derive(secret_test, 256)

print("Secret:", len(secret_test), "octets")
print("Compression OK:", decompresse(compresse_test) == texte_test)
print("Cle derivee:", len(cle_test), "octets")

Secret: 24 octets
Compression OK: True
Cle derivee: 32 octets


## 3. Chiffrement symétrique

On écrit les fonctions `encAES` et `decAES` analogues à celles du **TD2** pour chiffrer et déchiffrer un texte en utilisant le chiffrement `AES-256-GCM` pour lequel vous pourrez vous inspirer de la [documentation de la librairie Cryptography](https://cryptography.io/en/latest/hazmat/primitives/aead/) et la [page suivante](https://developers.google.com/tink/aead?hl=fr) pour le concept AEAD.

**Exercice 11.** Ecrivez la fonction `encAES` qui prend en entrée un texte au format `utf-8`, l'identité de l'expéditeur en donnée associée et un secret de 96 bits. Elle va successivement compresser le texte avec la fonction `compresse`, calculer une clé de session de 256 bits dérivée par la fonction `derive`, construire un nonce de 96 bits et chiffrer le compressé avec `AES-256-GCM`. 

In [100]:
def encAES(texte:str,clairauth:bytes,secret:bytes)->bytes:

**Exercice 12.** Ecrivez la fonction `desAES` qui inverse le fonctionnement de la fonction de chiffrement et retournera le clair au format `utf-8`.

In [102]:
def decAES(cryptogramme:bytes,clairauth:bytes,secret:bytes)->str:

## 4. Chiffrement hybride

### 4.1 Chiffrement hybride pour un destinataire

On veut implémenter un chiffrement hybride à la `PGP` pour lequel on va chiffrer par RSA le secret initial utilisé  par `AES` pour chiffrer un message. 
<br>


**Exercice 13.** Ecrivez la fonction de chiffrement `chiffre` qui prend en entrée un texte clair, la donnée associée (l'expéditeur) et la clé publique du destinataire et qui va fournir la sérialisation :

- du secret initial chiffré avec la clé publique RSA du destinataire dans une enveloppe digitale;
- du chiffré par `AES-256-GCM` à partir de la dérivation du secret initial de 96 bits;
- de la donnée associée.

Quelques précisions sur le format du chiffré hybride, au format `bytestream` qui concatène:
- la suite de 12 octets du secret initial est chiffrée par RSA avec la clé publique du destinataire;
- le chiffré;
- la donnée associée.

Ecrit en notation "Alice et Bob", en notant $S$ le secret initial, $pk$ la clé publique du destinataire et $m$ le message, on aura: $$\{S\}_{pk}.\{m\}_{\mbox{\scriptsize{kdf}}(S)}.\mbox{ad}$$
où $.$ dénote l'opération de concaténation, `kdf` la fonction de dérivation et `ad` la donnée associée.

Pratiquement, ces valeurs seront sérialisées au moyen de la librairie `pickle` et l'ensemble est ensuite converti au format `base64` et retourné à l'utilisateur.

In [104]:
def chiffre(message:str, aut:bytes, pk:rsa.RSAPublicKey)->bytes:

**Exercice 14.** Ecrivez la fonction `dechiffre` qui prend en entrée le chiffré hybride au format `base64` et la clé privée. Elle va décoder la suite sérialisée fournie au format `base64` pour retrouver les suites d'octets:
- du secret initial dans l'enveloppe digitale à déchiffrer pour récupérer le secret initial;
- du texte chiffré par `AES-256-GCM` à déchiffrer avec le secret initial récupéré;
- de la donnée associée.

In [109]:
def dechiffre(chiffre64:bytes, sk:rsa.RSAPrivateKey)->str:

### 4.2 Chiffrement hybride à destinataires multiples

On généralise le chiffrement hybride pour envoyer un même chiffré à une liste de destinataires. Pour cela, on fait appel à la fonction `chiffre` précédente. Le chiffrement est fait de la même façon mais on effectue la concaténation des enveloppes digitales par destinataire de la liste.

En notation Alice et Bob, en supposant avoir $n$ destinataires de clés publiques $pk_i, 1\leq i\leq n$, le message sera:

$$\{K\}_{pk_1}.\{K\}_{pk_2}...\{K\}_{pk_n}\{m\}_{\mbox{\scriptsize{kdf}}(K)}.\mbox{ad} $$

**Exercice 15.** Ecrivez la fonction `HMenc` qui prend en entrée un texte clair, la donnée associée et la liste des certificats des destinataires et qui va fournir la concaténation :

- de la liste des enveloppes digitales contenant les chiffrés du secret initial par les clés publiques RSA des destinataires;
- le chiffré du clair par `AES-256-GCM` utilisant le secret initial de 96 bits;
- la donnée associée.

La sortie sera recodée au format `base64` pour en assurer le transport.

In [110]:
def HMenc(message:str, aut:bytes, dest:list)->bytes:

**Exercice 16.** Ecrivez la fonction `HMdec` qui va déchiffrer le message pour un utilisateur sur présentation de sa clé privée. Si un autre utilisateur cherche à déchiffrer le message, il recevra le message `erreur: vous n'êtes pas destinataire`.

In [114]:
def HMdec(chiffre64:bytes, sk:rsa.RSAPrivateKey)->str:

### 4.3 Signature par l'expéditeur

Pour avoir une garantie sur l'authenticité de l'expéditeur, on signe le message par RSA, en s'inspirant du **TD4**.

**Exercice 17.** Ecrivez la fonction `sigRSA` qui prend en entrée une chaîne au format `bytestream`, la clé privée du signataire et qui retourne la signature au format `base64`.

In [116]:
def sigRSA(message:bytes, sk:rsa.RSAPrivateKey)->bytes:

**Exercice 18.** Ecrivez la fonction `verifRSA` qui prend en entrée la chaîne au format `bytestream`, sa signature au format `base64` et la clé publique du signataire. Elle retourne `verification OK` si la signature est valide et lève une exception sinon.

In [118]:
def verifRSA(message:bytes, signature:bytes, pk:rsa.RSAPublicKey)->str:

**Exercice 19.** Ecrivez les fonctions `HMencS` et `HMdecS` qui ajoutent la gestion d'une signature à `HMenc` et `HMdec`. La signature est sérialisée à la fin, après la concaténation de la liste des enveloppes digitales, du chiffré et du tag de données additionnelles.

In [120]:
def HMencS(message:str, aut:bytes, exp:rsa.RSAPrivateKey, dest:list)->bytes:

def HMdecS(chiffre64:bytes,exp:rsa.RSAPublicKey, sk:rsa.RSAPrivateKey)->str:

## 5. Usurpation d'identité

Le certificat de Xavier vous est fourni. Expliquez comment vous pouvez usurper son identité et appliquez votre méthode pour construire un message authentique qui provient de Xavier. Décrivez bien toutes les étapes !